# HAM10000 preprocessing only
This notebook reads your existing train/val/test CSVs without changing them. No model is trained. Attach the HAM10000 dataset and place your split CSVs in `/kaggle/working/splits`. Add the repository as a Kaggle input or copy it into `/kaggle/working`. Select a Python notebook; a GPU is optional.

In [ ]:
from pathlib import Path
import sys

# EDIT this to the folder containing ham10000/ in your attached repository.
PROJECT_ROOT = Path("/kaggle/working/Hybrid-CNN-ViT-Skin-Lesion-Classification")
if not (PROJECT_ROOT / "ham10000").is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to the repository folder containing ham10000/")
sys.path.insert(0, str(PROJECT_ROOT))


## Validate paths, labels, and lesion separation
CSVs require `image_id`. Labels (`dx` or `label`) and `lesion_id` can come from each CSV or from `HAM10000_metadata.csv`. Optional `image_path` values may be absolute or relative to the dataset root. Class indices are fixed in the order shown below. Validation fully decodes images and may take a few minutes. Errors stop execution; fix the upstream issue rather than modifying splits here.

In [ ]:
from ham10000.data import CLASS_TO_INDEX
from ham10000.dataset import create_dataloaders

DATA_ROOT = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
SPLIT_ROOT = "/kaggle/working/splits"
print("Class mapping:", CLASS_TO_INDEX)
loaders, report = create_dataloaders(
    DATA_ROOT, SPLIT_ROOT, batch_size=16, num_workers=2, seed=42
)
import pandas as pd
display(pd.DataFrame({name: info["class_counts"] for name, info in report.items()}))
for name, info in report.items():
    print(name, "images:", info["images"], "lesions:", info["lesions"])


## Inspect one batch (no training)
Only training uses random flips, rotation, and mild color jitter. Every image becomes an RGB float tensor of shape `[3, 224, 224]` with ImageNet normalization. Validation/test are deterministic and keep CSV order. Final partial batches are retained. Set `num_workers=0` above if notebook workers fail.

In [ ]:
import torch
for name, loader in loaders.items():
    images, labels = next(iter(loader))
    assert images.shape[1:] == (3, 224, 224)
    assert labels.dtype == torch.long
    assert torch.isfinite(images).all()
    assert ((labels >= 0) & (labels < 7)).all()
    print(name, "image shape:", tuple(images.shape), "labels:", labels.tolist())


The preprocessing pipeline is ready when validation and batch inspection complete. This notebook does not fit a model, rebalance classes, or write CSVs/images. Multiple images of one lesion are allowed within a split, but never across splits. Missing classes are reported as zero counts. A seed reproduces fresh loader runs; repeated training iterations intentionally get new augmentations.